## Explainability — NumpyGPT

In [ ]:
import os
import rootutils
import numpy as np
import matplotlib.pyplot as plt

rootutils.setup_root(os.path.abspath(''), indicator=['.git', 'pyproject.toml'], pythonpath=True)

from src.utils import load_encoder_hparams_and_params
from src.gpt2 import gpt2
from src.visualize import collect_attention, plot_attention, plot_top_k_logits

PROMPT = 'Computers can help'
encoder, hparams, params = load_encoder_hparams_and_params('124M', '../models')
n_head = hparams['n_head']

input_ids = encoder.encode(PROMPT)
tokens = [encoder.decode([i]) for i in input_ids]
print('Tokens:', tokens)

---
### Attention Heatmaps

`collect_attention()` captures the softmax weights `[n_head, n_seq, n_seq]` from every transformer block during a single forward pass.
Each cell `[h, i, j]` is how much token `i` attends to token `j` in head `h`.

In [ ]:
with collect_attention() as attn_weights:
    logits = gpt2(input_ids, **params, n_head=n_head)

print(f'Collected {len(attn_weights)} layers')
print(f'Each layer shape: {attn_weights[0].shape}  (n_head, n_seq, n_seq)')

In [ ]:
# All 12 heads for layer 0
plot_attention(attn_weights, tokens, layer=0)

In [ ]:
# Compare early vs late layers — early layers tend to attend locally, late layers more globally
plot_attention(attn_weights, tokens, layer=5)
plot_attention(attn_weights, tokens, layer=11)

---
### Top-k Next-Token Probabilities

`logits[-1]` is the model's output distribution over the vocabulary for the next token.
This shows what the model considers before committing to the greedy argmax pick.

In [ ]:
plot_top_k_logits(logits[-1], encoder, k=10)

In [ ]:
# Top-5 predictions at every position — what the model predicted after seeing each prefix
fig, axes = plt.subplots(1, len(tokens), figsize=(4 * len(tokens), 3))
for pos, (ax, tok) in enumerate(zip(axes, tokens)):
    probs = np.exp(logits[pos] - np.max(logits[pos]))
    probs /= probs.sum()
    top_ids = np.argsort(probs)[-5:][::-1]
    ax.barh(range(5), probs[top_ids][::-1])
    ax.set_yticks(range(5))
    ax.set_yticklabels([repr(encoder.decode([int(i)])) for i in top_ids[::-1]], fontsize=8)
    ax.set_title(f'after {repr(tok)}', fontsize=8)
    ax.set_xlabel('prob', fontsize=7)
plt.tight_layout()
plt.show()